In [1]:

import pandas as pd

FILE_NAME = "Bus Planning.xlsx"
START_CHARGE = 400.0
MAX_CHARGE = 400.0


def track_bus_charges(file_name):
    # Read the planning Excel file
    df = pd.read_excel(file_name)

    # Use the columns directly
    bus_col = "bus"
    activity_col = "activity"
    energy_col = "energy consumption"

    # Convert energy values to numbers
    df[energy_col] = pd.to_numeric(
        df[energy_col],
        errors="coerce"
    ).fillna(0)

    # Loop through each bus
    for bus, bus_rows in df.groupby(bus_col, sort=False):
        charge = START_CHARGE
        charges = [charge]

        # Loop through each planning row
        for _, row in bus_rows.iterrows():
            energy = float(row[energy_col])
            activity = str(row[activity_col]).strip().lower()

            # Charging adds energy
            if "charg" in activity:
                charge += abs(energy)
            else:
                # Driving / other activity consumes energy
                charge -= energy

            # Keep battery between 0 and 400 kWh
            charge = max(0.0, min(charge, MAX_CHARGE))

            charges.append(charge)

        # Print results
        print(f"Bus {bus}:")
        print(" - ".join(f"{c:.2f} KWh" for c in charges))
        print()


track_bus_charges(FILE_NAME)


Bus 1:
400.00 KWh - 398.02 KWh - 387.22 KWh - 382.22 KWh - 371.36 KWh - 366.36 KWh - 355.55 KWh - 350.55 KWh - 339.69 KWh - 334.69 KWh - 321.84 KWh - 316.84 KWh - 305.98 KWh - 295.66 KWh - 284.80 KWh - 274.48 KWh - 263.62 KWh - 253.30 KWh - 242.44 KWh - 232.12 KWh - 221.26 KWh - 210.94 KWh - 200.08 KWh - 189.76 KWh - 178.90 KWh - 168.58 KWh - 157.72 KWh - 147.40 KWh - 136.54 KWh - 294.04 KWh - 296.02 KWh - 285.22 KWh - 274.36 KWh - 264.04 KWh - 253.18 KWh - 240.33 KWh - 235.33 KWh - 230.33 KWh - 219.47 KWh - 209.15 KWh - 198.29 KWh - 187.97 KWh - 177.11 KWh - 166.79 KWh - 155.93 KWh - 150.93 KWh - 140.13 KWh - 135.13 KWh - 124.27 KWh - 122.29 KWh - 332.29 KWh - 330.31 KWh - 319.51 KWh - 314.51 KWh - 303.65 KWh - 298.65 KWh - 287.84 KWh - 282.84 KWh - 271.98 KWh - 266.98 KWh - 256.18 KWh - 245.32 KWh - 240.32 KWh - 238.34 KWh

Bus 2:
400.00 KWh - 398.02 KWh - 387.22 KWh - 382.22 KWh - 371.36 KWh - 360.55 KWh - 355.55 KWh - 350.55 KWh - 339.69 KWh - 334.69 KWh - 321.84 KWh - 316.84 KWh -

In [8]:
import pandas as pd
import os

FILE_NAME = "Bus Planning.xlsx"

START_CHARGE = 400.0
MAX_CHARGE = 400.0

REQUIRED_COLUMNS = [
    "start location",
    "end location",
    "start time",
    "end time",
    "activity",
    "line",
    "energy consumption",
    "bus"
]

ALLOWED_LOCATIONS = [
    "ehvbst",
    "ehvapt",
    "ehvgar"
]

ALLOWED_ACTIVITIES = [
    "idle",
    "service trip",
    "material trip",
    "charging"
]

ALLOWED_LINES = [
    400,
    401
]


def time_to_minutes(time_value):
    """
    Convert a time into minutes since midnight.

    Example:
    23:40 -> 1420
    00:10 -> 10
    """
    time_value = pd.to_datetime(time_value)

    return (
        time_value.hour * 60
        + time_value.minute
    )


def read_and_check_file(file_name):
    # ---------------------------------------------------------
    # 1. Check file type
    # ---------------------------------------------------------
    ext = os.path.splitext(file_name)[1].lower()

    if ext not in [".xlsx", ".xls"]:
        raise ValueError(
            "Your file type is invalid. "
            "Please input an Excel file (.xlsx or .xls)."
        )

    # ---------------------------------------------------------
    # 2. Try to read the file
    # ---------------------------------------------------------
    try:
        df = pd.read_excel(file_name)

    except Exception as e:
        raise ValueError(
            f"The file could not be read. Please check that the "
            f"file is a valid Excel file.\nOriginal error: {e}"
        )

    # ---------------------------------------------------------
    # 3. Check required columns
    # ---------------------------------------------------------
    missing_columns = [
        column for column in REQUIRED_COLUMNS
        if column not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Unable to proceed. Missing columns: {missing_columns}"
        )

    # ---------------------------------------------------------
    # 4. Check locations
    # ---------------------------------------------------------
    for idx, row in df.iterrows():

        start_location = (
            str(row["start location"])
            .strip()
            .lower()
        )

        end_location = (
            str(row["end location"])
            .strip()
            .lower()
        )

        activity = (
            str(row["activity"])
            .strip()
            .lower()
        )

        if start_location not in ALLOWED_LOCATIONS:
            raise ValueError(
                f"Row {idx + 2}: Invalid start location "
                f"'{row['start location']}'. "
                f"Allowed values are: {ALLOWED_LOCATIONS}"
            )

        if end_location not in ALLOWED_LOCATIONS:
            raise ValueError(
                f"Row {idx + 2}: Invalid end location "
                f"'{row['end location']}'. "
                f"Allowed values are: {ALLOWED_LOCATIONS}"
            )

        # Idle and charging are allowed to have the same
        # start and end location
        if (
            start_location == end_location
            and activity not in ["idle", "charging"]
        ):
            raise ValueError(
                f"Row {idx + 2}: Start and end locations "
                f"cannot be the same."
            )

    # ---------------------------------------------------------
    # 5. Check activities
    # ---------------------------------------------------------
    for idx, value in df["activity"].items():

        if pd.isna(value) or str(value).strip() == "":
            print(
                f"Warning: Row {idx + 2} has an empty activity."
            )
            continue

        activity = (
            str(value)
            .strip()
            .lower()
        )

        if activity not in ALLOWED_ACTIVITIES:
            raise ValueError(
                f"Row {idx + 2}: Invalid activity "
                f"'{value}'. Allowed values are: "
                f"{ALLOWED_ACTIVITIES}"
            )

    # ---------------------------------------------------------
    # 6. Check energy consumption
    # ---------------------------------------------------------
    for idx, value in df["energy consumption"].items():

        energy = pd.to_numeric(
            value,
            errors="coerce"
        )

        if pd.isna(energy):
            raise ValueError(
                f"Row {idx + 2}: '{value}' in "
                f"'energy consumption' is not a number."
            )

        activity = (
            str(df.loc[idx, "activity"])
            .strip()
            .lower()
        )

        # Idle must consume exactly 5
        if activity == "idle" and energy != 5:
            raise ValueError(
                f"Row {idx + 2}: Idle activities must "
                f"consume exactly 5 energy."
            )

        # Charging must have negative energy
        if activity == "charging" and energy >= 0:
            raise ValueError(
                f"Row {idx + 2}: Charging must have "
                f"a negative energy consumption value."
            )

        # Non-charging activities must consume positive energy
        if activity != "charging" and energy < 0:
            raise ValueError(
                f"Row {idx + 2}: Energy consumption must "
                f"be positive for non-charging activities."
            )

    # ---------------------------------------------------------
    # 7. Check service-trip line requirement
    # ---------------------------------------------------------
    for idx, row in df.iterrows():

        activity = (
            str(row["activity"])
            .strip()
            .lower()
        )

        line = row["line"]

        if activity == "service trip":

            if pd.isna(line) or str(line).strip() == "":
                raise ValueError(
                    f"Row {idx + 2}: A service trip must "
                    f"have a line."
                )

            if line not in ALLOWED_LINES:
                raise ValueError(
                    f"Row {idx + 2}: Invalid line '{line}'. "
                    f"Allowed lines are: {ALLOWED_LINES}"
                )

        else:
            # Non-service-trip lines are ignored
            continue

    # ---------------------------------------------------------
    # 8. Check start/end times
    # ---------------------------------------------------------

    # Convert Excel times to pandas datetime
    df["start time"] = pd.to_datetime(
        df["start time"],
        errors="coerce"
    )

    df["end time"] = pd.to_datetime(
        df["end time"],
        errors="coerce"
    )

    # Create columns containing minutes since midnight
    df["start minutes"] = None
    df["end minutes"] = None
    df["duration minutes"] = None

    for idx, row in df.iterrows():

        start_time = row["start time"]
        end_time = row["end time"]

        if pd.isna(start_time) or pd.isna(end_time):
            raise ValueError(
                f"Row {idx + 2}: Invalid or missing "
                f"start/end time."
            )

        # Convert times to minutes since midnight
        start_minutes = time_to_minutes(start_time)
        end_minutes = time_to_minutes(end_time)

        activity = (
            str(row["activity"])
            .strip()
            .lower()
        )

        # Calculate duration
        duration = end_minutes - start_minutes

        # If the end time is earlier than the start time,
        # assume the activity crosses midnight.
        #
        # Example:
        # 23:40 -> 00:10
        #
        # 10 - 1420 = -1410
        # -1410 + 1440 = 30 minutes
        if duration < 0:
            duration += 1440

        # Start and end being exactly the same is allowed
        # for idle, but not for other activities.
        if duration == 0 and activity != "idle":
            raise ValueError(
                f"Row {idx + 2}: Start time and end time "
                f"cannot be the same."
            )

        # Store the values
        df.loc[idx, "start minutes"] = start_minutes
        df.loc[idx, "end minutes"] = end_minutes
        df.loc[idx, "duration minutes"] = duration

    # ---------------------------------------------------------
    # 9. Check that a bus cannot make overlapping trips
    # ---------------------------------------------------------

    # Add original row order so we do not lose the
    # planning order from Excel.
    df["_planning_order"] = range(len(df))

    for bus, bus_rows in df.groupby("bus", sort=False):

        bus_rows = bus_rows.sort_values("_planning_order")

        previous_start = None
        previous_end = None

        for idx, row in bus_rows.iterrows():

            start = int(row["start minutes"])
            end = start + int(row["duration minutes"])

            # If this activity occurs after midnight,
            # move it forward relative to the previous activity.
            if previous_start is not None:

                while start < previous_start:
                    start += 1440
                    end += 1440

            # Check for overlap
            if (
                previous_end is not None
                and start < previous_end
            ):
                raise ValueError(
                    f"Bus {bus} has overlapping trips "
                    f"around row {idx + 2}."
                )

            previous_start = start
            previous_end = end

    return df


def track_bus_charges(file_name):

    # Read and validate the planning file
    df = read_and_check_file(file_name)

    # Loop through each bus
    for bus, bus_rows in df.groupby(
        "bus",
        sort=False
    ):

        bus_rows = bus_rows.sort_values(
            "_planning_order"
        )

        charge = START_CHARGE
        charges = [charge]

        # Process each planning row
        for _, row in bus_rows.iterrows():

            energy = float(
                row["energy consumption"]
            )

            activity = (
                str(row["activity"])
                .strip()
                .lower()
            )

            # Charging adds energy because the input value
            # is negative.
            if activity == "charging":
                charge += abs(energy)

            else:
                # Driving / idle / other activity consumes energy
                charge -= energy

            # Keep battery between 0 and 400 kWh
            charge = max(
                0.0,
                min(
                    charge,
                    MAX_CHARGE
                )
            )

            charges.append(charge)

        # Print results
        print(f"Bus {bus}:")
        print(
            " - ".join(
                f"{c:.2f} KWh"
                for c in charges
            )
        )
        print()


# ---------------------------------------------------------
# Run the program
# ---------------------------------------------------------
track_bus_charges(FILE_NAME)



C:\Users\ivanm\AppData\Local\Temp\ipykernel_35788\784915193.py:243: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["start time"] = pd.to_datetime(
C:\Users\ivanm\AppData\Local\Temp\ipykernel_35788\784915193.py:248: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["end time"] = pd.to_datetime(


ValueError: Bus 1 has overlapping trips around row 37.

In [15]:

import pandas as pd
import os


# =========================================================
# SETTINGS
# =========================================================

FILE_NAME = "Bus Planning.xlsx"

START_CHARGE = 400.0
MAX_CHARGE = 400.0

REQUIRED_COLUMNS = [
    "start location",
    "end location",
    "start time",
    "end time",
    "activity",
    "line",
    "energy consumption",
    "bus"
]

ALLOWED_LOCATIONS = [
    "ehvbst",
    "ehvapt",
    "ehvgar"
]

ALLOWED_ACTIVITIES = [
    "idle",
    "service trip",
    "material trip",
    "charging"
]

ALLOWED_LINES = [
    400,
    401
]


# =========================================================
# TIME FUNCTION
# =========================================================

def time_to_minutes(time_value):
    """
    Convert a time into minutes since midnight.

    Examples:
        23:40 -> 1420
        00:10 -> 10
        08:30 -> 510
    """

    time_value = pd.to_datetime(time_value)

    return (
        time_value.hour * 60
        + time_value.minute
    )


# =========================================================
# READ AND VALIDATE FILE
# =========================================================

def read_and_check_file(file_name):

    # -----------------------------------------------------
    # 1. Check file type
    # -----------------------------------------------------

    ext = os.path.splitext(file_name)[1].lower()

    if ext not in [".xlsx", ".xls"]:
        raise ValueError(
            "Your file type is invalid. "
            "Please input an Excel file (.xlsx or .xls)."
        )

    # -----------------------------------------------------
    # 2. Try to read the file
    # -----------------------------------------------------

    try:

        df = pd.read_excel(file_name)

    except Exception as e:

        raise ValueError(
            f"The file could not be read. Please check that "
            f"the file is a valid Excel file.\n"
            f"Original error: {e}"
        )

    # -----------------------------------------------------
    # 3. Check required columns
    # -----------------------------------------------------

    missing_columns = [
        column
        for column in REQUIRED_COLUMNS
        if column not in df.columns
    ]

    if missing_columns:

        raise ValueError(
            f"Unable to proceed. Missing columns: "
            f"{missing_columns}"
        )

    # -----------------------------------------------------
    # 4. Check locations
    # -----------------------------------------------------

    for idx, row in df.iterrows():

        start_location = (
            str(row["start location"])
            .strip()
            .lower()
        )

        end_location = (
            str(row["end location"])
            .strip()
            .lower()
        )

        activity = (
            str(row["activity"])
            .strip()
            .lower()
        )

        # Check start location

        if start_location not in ALLOWED_LOCATIONS:

            raise ValueError(
                f"Row {idx + 2}: Invalid start location "
                f"'{row['start location']}'. "
                f"Allowed values are: {ALLOWED_LOCATIONS}"
            )

        # Check end location

        if end_location not in ALLOWED_LOCATIONS:

            raise ValueError(
                f"Row {idx + 2}: Invalid end location "
                f"'{row['end location']}'. "
                f"Allowed values are: {ALLOWED_LOCATIONS}"
            )

        # Start and end locations can only be the same
        # for idle and charging activities.

        if (
            start_location == end_location
            and activity not in ["idle", "charging"]
        ):

            raise ValueError(
                f"Row {idx + 2}: Start and end locations "
                f"cannot be the same."
            )

    # -----------------------------------------------------
    # 5. Check activities
    # -----------------------------------------------------

    for idx, value in df["activity"].items():

        if pd.isna(value) or str(value).strip() == "":

            print(
                f"Warning: Row {idx + 2} has an empty activity."
            )

            continue

        activity = (
            str(value)
            .strip()
            .lower()
        )

        if activity not in ALLOWED_ACTIVITIES:

            raise ValueError(
                f"Row {idx + 2}: Invalid activity "
                f"'{value}'. "
                f"Allowed values are: "
                f"{ALLOWED_ACTIVITIES}"
            )

    # -----------------------------------------------------
    # 6. Check energy consumption
    # -----------------------------------------------------

    for idx, value in df["energy consumption"].items():

        energy = pd.to_numeric(
            value,
            errors="coerce"
        )

        if pd.isna(energy):

            raise ValueError(
                f"Row {idx + 2}: '{value}' in "
                f"'energy consumption' is not a number."
            )

        activity = (
            str(df.loc[idx, "activity"])
            .strip()
            .lower()
        )

        # Idle must consume exactly 5 kWh

        if activity == "idle" and energy != 5:

            raise ValueError(
                f"Row {idx + 2}: Idle activities must "
                f"consume exactly 5 energy."
            )

        # Charging must have negative energy

        if activity == "charging" and energy >= 0:

            raise ValueError(
                f"Row {idx + 2}: Charging must have "
                f"a negative energy consumption value."
            )

        # Non-charging activities must have positive energy

        if activity != "charging" and energy < 0:

            raise ValueError(
                f"Row {idx + 2}: Energy consumption must "
                f"be positive for non-charging activities."
            )

    # -----------------------------------------------------
    # 7. Check service-trip line requirement
    # -----------------------------------------------------

    for idx, row in df.iterrows():

        activity = (
            str(row["activity"])
            .strip()
            .lower()
        )

        line = row["line"]

        # Service trips must have a valid line

        if activity == "service trip":

            if pd.isna(line) or str(line).strip() == "":

                raise ValueError(
                    f"Row {idx + 2}: A service trip must "
                    f"have a line."
                )

            if line not in ALLOWED_LINES:

                raise ValueError(
                    f"Row {idx + 2}: Invalid line '{line}'. "
                    f"Allowed lines are: {ALLOWED_LINES}"
                )

        # Lines for non-service activities are ignored.

    # -----------------------------------------------------
    # 8. Check start and end times
    # -----------------------------------------------------

    df["start time"] = pd.to_datetime(
        df["start time"],
        errors="coerce"
    )

    df["end time"] = pd.to_datetime(
        df["end time"],
        errors="coerce"
    )

    # Create helper columns

    df["start minutes"] = None
    df["end minutes"] = None
    df["duration minutes"] = None

    for idx, row in df.iterrows():

        start_time = row["start time"]
        end_time = row["end time"]

        # Check that both times exist

        if pd.isna(start_time) or pd.isna(end_time):

            raise ValueError(
                f"Row {idx + 2}: Invalid or missing "
                f"start/end time."
            )

        # Convert to minutes since midnight

        start_minutes = time_to_minutes(start_time)
        end_minutes = time_to_minutes(end_time)

        activity = (
            str(row["activity"])
            .strip()
            .lower()
        )

        # Calculate duration

        duration = end_minutes - start_minutes

        # If end time is earlier than start time,
        # assume the activity crosses midnight.

        if duration < 0:

            duration += 1440

        # Same start/end time is allowed only for idle.

        if duration == 0 and activity != "idle":

            raise ValueError(
                f"Row {idx + 2}: Start time and end time "
                f"cannot be the same."
            )

        # Store calculated values

        df.loc[idx, "start minutes"] = start_minutes
        df.loc[idx, "end minutes"] = end_minutes
        df.loc[idx, "duration minutes"] = duration

    # -----------------------------------------------------
    # 9. Preserve original planning order
    # -----------------------------------------------------

    df["_planning_order"] = range(len(df))

    # -----------------------------------------------------
    # 10. Check overlapping activities
    # -----------------------------------------------------

    for bus, bus_rows in df.groupby(
        "bus",
        sort=False
    ):

        bus_rows = bus_rows.sort_values(
            "_planning_order"
        )

        previous_start = None
        previous_end = None

        for idx, row in bus_rows.iterrows():

            start = int(row["start minutes"])

            end = (
                start
                + int(row["duration minutes"])
            )

            # If this activity occurs after midnight,
            # move it forward relative to the previous activity.

            if previous_start is not None:

                while start < previous_start:

                    start += 1440
                    end += 1440

            # Check for overlap

            if (
                previous_end is not None
                and start < previous_end
            ):

                raise ValueError(
                    f"Bus {bus} has overlapping trips "
                    f"around row {idx + 2}."
                )

            previous_start = start
            previous_end = end

    # -----------------------------------------------------
    # File has passed all validation checks
    # -----------------------------------------------------

    return df


# =========================================================
# BATTERY / FEASIBILITY CHECK
# =========================================================

def track_bus_charges(file_name):

    # -----------------------------------------------------
    # Read and validate the planning file
    # -----------------------------------------------------

    df = read_and_check_file(file_name)

    # -----------------------------------------------------
    # Process each bus
    # -----------------------------------------------------

    for bus, bus_rows in df.groupby(
        "bus",
        sort=False
    ):

        bus_rows = bus_rows.sort_values(
            "_planning_order"
        )

        # Every bus starts with a full battery.

        charge = START_CHARGE

        charges = [charge]

        # Process every activity for this bus


        for idx, row in bus_rows.iterrows():

            energy = float(
                row["energy consumption"]
            )

            activity = (
                str(row["activity"])
            )

            # =============================================
            # CHARGING
            # =============================================

            if activity == "charging":

                charging_energy = abs(energy)

                new_charge = (
                    charge
                    + charging_energy
                )

                # Check maximum battery capacity.

                if new_charge > MAX_CHARGE:

                    raise ValueError(
                        f"\n"
                        f"Battery feasibility error!\n"
                        f"Bus: {bus}\n"
                        f"Excel row: {idx + 2}\n"
                        f"Activity: {activity}\n"
                        f"Current battery: "
                        f"{charge:.2f} kWh\n"
                        f"Charging energy: "
                        f"{charging_energy:.2f} kWh\n"
                        f"Resulting battery: "
                        f"{new_charge:.2f} kWh\n"
                        f"Maximum battery capacity: "
                        f"{MAX_CHARGE:.2f} kWh\n"
                    )

                # Charging is feasible.

                charge = new_charge

            # =============================================
            # ENERGY CONSUMING ACTIVITY
            # =============================================

            else:

                new_charge = (
                    charge
                    - energy
                )

                # Check whether the bus has eno energy to perform the activity.

                if new_charge < 0:

                    energy_shortfall = abs(new_charge)

                    raise ValueError(
                        f"\n"
                        f"Battery feasibility error!\n"
                        f"Bus: {bus}\n"
                        f"Excel row: {idx + 2}\n"
                        f"Activity: {activity}\n"
                        f"Current battery: "
                        f"{charge:.2f} kWh\n"
                        f"Required energy: "
                        f"{energy:.2f} kWh\n"
                        f"Energy shortfall: "
                        f"{energy_shortfall:.2f} kWh\n"
                    )

                # Activity is feasible.

                charge = new_charge

            # Store battery level after activity.

            charges.append(charge)

        # Print battery results


        print(f"Bus {bus}:")

        print(
            " - ".join(
                f"{c:.2f} KWh"
                for c in charges
            )
        )

        print()

    # -----------------------------------------------------
    # If we reach this point every bus should be feasible.
    # -----------------------------------------------------


    print("BUS PLANNING RESULT")

    print("The bus schedule is FEASIBLE.")
    print("All battery constraints have been satisfied.")


# =========================================================
# RUN PROGRAM
# =========================================================

try:

    track_bus_charges(FILE_NAME)

except ValueError as e:


    print("BUS PLANNING RESULT")

    print("The bus schedule is NOT FEASIBLE.")

    print(e)


C:\Users\ivanm\AppData\Local\Temp\ipykernel_35788\329353018.py:292: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["start time"] = pd.to_datetime(
C:\Users\ivanm\AppData\Local\Temp\ipykernel_35788\329353018.py:297: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["end time"] = pd.to_datetime(


Bus 1:
400.00 KWh - 398.02 KWh - 387.22 KWh - 382.22 KWh - 371.36 KWh - 366.36 KWh - 355.55 KWh - 350.55 KWh - 339.69 KWh - 334.69 KWh - 321.84 KWh - 316.84 KWh - 305.98 KWh - 295.66 KWh - 284.80 KWh - 274.48 KWh - 263.62 KWh - 253.30 KWh - 242.44 KWh - 232.12 KWh - 221.26 KWh - 210.94 KWh - 200.08 KWh - 189.76 KWh - 178.90 KWh - 168.58 KWh - 157.72 KWh - 147.40 KWh - 136.54 KWh - 134.56 KWh - 292.06 KWh - 281.26 KWh - 270.40 KWh - 260.08 KWh - 249.22 KWh - 236.37 KWh - 231.37 KWh - 220.51 KWh - 210.19 KWh - 199.33 KWh - 189.01 KWh - 178.15 KWh - 167.83 KWh - 156.97 KWh - 151.97 KWh - 141.17 KWh - 136.17 KWh - 125.31 KWh - 123.33 KWh - 333.33 KWh - 331.35 KWh - 320.55 KWh - 315.55 KWh - 304.69 KWh - 299.69 KWh - 288.88 KWh - 283.88 KWh - 273.02 KWh - 268.02 KWh - 257.22 KWh - 246.36 KWh - 244.38 KWh

Bus 2:
400.00 KWh - 398.02 KWh - 387.22 KWh - 382.22 KWh - 371.36 KWh - 360.55 KWh - 355.55 KWh - 344.69 KWh - 331.84 KWh - 326.84 KWh - 315.98 KWh - 305.66 KWh - 294.80 KWh - 284.48 KWh -

In [1]:
import pandas as pd

df = pd.read_excel("Cleaned Bus Planning.xlsx")
x = "line"
line = None
for i in range(len(df) -1, -1, -1):
    if pd.notna(df.loc[i, x]):
        line = df.loc[i, x]
    else:
        df.loc[i, x] = line
df.to_excel("Lines fixed.xlsx", index=False)

FileNotFoundError: [Errno 2] No such file or directory: 'Cleaned Bus Planning.xlsx'